# Numerical optimisation

How can I walk a parameter so that a loss gets smaller?

A fit chooses a parameter by making a loss as small as possible. For a line through the origin, $y = ax$, one loss is the sum of squared errors:

$$
\mathrm{SSE}(a) = \sum_i (y_i - a x_i)^2
$$

The points here are $x = 1, 2, 3, 4$ and $y = 1, 2, 3, 5$. The next cell evaluates the loss at three slopes. The smallest of those three is only the winner among the slopes that were tried.


In [ ]:
# The sum of squared misses for a few slopes through the origin. The line has no intercept, so a steeper slope lifts every prediction.
# The inputs.
xs = [1, 2, 3, 4]
# The outcomes.
ys = [1, 2, 3, 5]

# Sum of squared misses for a slope a.
def sse(a):
    # A running total of the squared misses.
    total = 0
    # One point at a time.
    for i in range(4):
        total = total + (ys[i] - a * xs[i]) ** 2
    return total

# Three slopes to compare.
for a in [1, 1.1, 1.5]:
    # The slope, and its sum of squares.
    print(a, round(sse(a), 3))


## A downhill walk

The slope of the loss says which way is up. A step the other way is

$$
a_{\text{new}} = a - \text{step} \times (\text{slope of the loss})
$$

The next cell starts at 0 and takes five steps of size 0.02. The first step goes past the bottom. The later steps come back. Run the cell. A player is drawn under the figure. Press play, or drag the slider. The first plot in a session can take a moment.


In [ ]:
# Walk downhill on that sum of squares, and follow the walk on the curve. Each step moves opposite the slope of the loss.
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# The same inputs as the cell above.
xs = [1, 2, 3, 4]
# The same outcomes.
ys = [1, 2, 3, 5]

# The loss: sum of squared misses at slope a.
def sse(a):
    # Running total inside the loss.
    total = 0
    # One point at a time, for the loss.
    for i in range(4):
        total = total + (ys[i] - a * xs[i]) ** 2
    return total

# How steeply the loss is rising at this slope.
def slope_of_loss(a):
    # A fresh total for the slope formula.
    total = 0
    # One point at a time, for the slope of the loss.
    for i in range(4):
        total = total + xs[i] * (ys[i] - a * xs[i])
    return -2 * total

# The walk starts at a slope of 0.
path = [0.0]
# Five downhill steps.
for i in range(5):
    # Step opposite the slope. 0.02 sets how far one unit of slope moves you.
    path.append(path[-1] - 0.02 * slope_of_loss(path[-1]))
# The slope at the start and after each step.
print([round(a, 3) for a in path])

# Slopes from 0 to 2, for drawing the loss curve.
grid = [i / 20 for i in range(41)]
# The figure the player draws into.
fig, ax = plt.subplots(figsize=(6.4, 3.4))
# The loss curve, in grey.
ax.plot(grid, [sse(a) for a in grid], color="0.65")
# The walker, starting off the figure until the first frame.
point, = ax.plot([], [], marker="o")
# Name the horizontal axis.
ax.set_xlabel("Slope")
# Name the vertical axis.
ax.set_ylabel("Sum of squared errors")

# Put the walker on the curve at this step.
def draw(frame):
    # The slope at this step of the walk.
    a = path[frame]
    # One point: that slope, and the loss there.
    point.set_data([a], [sse(a)])
    ax.set_title("Step " + str(frame) + ": slope " + str(round(a, 3)))
    return (point,)

# Build a player under the figure. Press play, or drag the slider, to step through the frames.
anim = FuncAnimation(fig, draw, frames=len(path), interval=500, repeat=False)
# Close the static figure so only the player is shown.
plt.close(fig)
# The player under the cell.
HTML(anim.to_jshtml())


## A formula, and a library search

For this loss there is also a formula. It is the slope that minimises the sum of squared errors for a line through the origin:

$$
a = \frac{\sum_i x_i y_i}{\sum_i x_i^2}
$$

A library search does not need that formula. It starts at a guess and walks downhill, as the player did. The first time a cell imports `scipy`, that library downloads. Later cells in the session reuse it.

The next cell prints the formula, then the slope `scipy` finds from a start of 0. Rounded to three decimals, both are 1.133.


In [ ]:
# The slope from the formula for a line through the origin. It is the sum of each x times y, divided by the sum of each x squared.
# The inputs.
xs = [1, 2, 3, 4]
# The outcomes.
ys = [1, 2, 3, 5]

# Totals for the formula.
total_xy = 0
total_xx = 0
# Add each point.
for i in range(4):
    total_xy = total_xy + xs[i] * ys[i]
    total_xx = total_xx + xs[i] ** 2
# The formula's slope.
print(round(total_xy / total_xx, 3))

# Ask a library routine for the same slope, starting the search at 0. It should settle on the slope the formula just gave.
# A routine that searches for a low value of a function.
from scipy.optimize import minimize

# The loss the search will walk on. a is a short list, so the slope is its first entry.
def sse(a):
    # Running total of squared misses.
    total = 0
    # The slope being tried. minimize passes a list, not a bare number.
    slope = a[0]
    # One point at a time.
    for i in range(4):
        total = total + (ys[i] - slope * xs[i]) ** 2
    return total

# Search from a slope of 0.
result = minimize(sse, x0=[0.0])
# The slope the search settled on.
print(round(float(result.x[0]), 3))


## Check yourself

1. For $x = 1, 2, 3$ and $y = 2, 4, 5$, the least-squares slope through the origin, rounded to three decimals.
2. The sum of squared errors at that slope, rounded to three decimals.

Type your answer in place of `...` in the cell below, then run the cell. A match prints `That matches.` A first miss prints a hint. Run the cell again, or pass `show=True`, to see the expected result. The cell loads `check` from `check.py` in the file list.

If the blank is inside a function, include `return` so the function gives a value back.


In [ ]:
# Load check from check.py so this cell can compare what you type with the stored result.
# On the notebook page that file is at /drive. Opened from a lesson folder, it is in the parent folder.
import sys
from pathlib import Path

# Use /drive when the file is there. Otherwise use the parent folder.
if Path("/drive/check.py").is_file():
    root = "/drive"
elif (Path.cwd().parent / "check.py").is_file():
    root = str(Path.cwd().parent)
else:
    root = ""
# Add that folder to the places Python searches, then load check.
if root and root not in sys.path:
    sys.path.append(root)
from check import check

# Question 1: type your answer in place of .... This blank is the value the question asks for.
answer = ...
# Run this check. A match prints That matches. A first miss prints the hint.
check("mf01-1", answer, hint="Add each x*y, then divide by the sum of x squared. Round to three decimals.")
# Question 2: type your answer in place of .... This blank is the value the question asks for.
answer = ...
# Run this check. A match prints That matches. A first miss prints the hint.
check("mf01-2", answer, hint="Use the unrounded slope inside the errors, then round the sum of squares to three decimals.")


## What you practised

You computed a sum of squared errors for a line through the origin, and you stepped the slope against the slope of that loss. The same slope also comes from a formula and from a library search.
